In [ ]:
import os, math, gc
import numpy as np, pandas as pd, torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm.auto import tqdm

# ----------------- CONFIG -----------------
SEQ_LEN = 60
BATCH_SIZE = 256
DMODEL = 256
NUM_LAYERS = 4
NHEAD = 8
D_FF = 512
DROPOUT = 0.2
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
MODEL_PATH = "best_transformer_improved.pt"

# ----------------- LOAD DATA -----------------
train_df = pd.read_csv("training_price_features.csv")
test_df  = pd.read_csv("validation_price_features.csv")

exclude = ["Date", "Target", "RowId"]
FEATURES = [c for c in train_df.columns if c not in exclude]
keep_cols = FEATURES + ["Target"]

train_df = train_df.dropna(subset=keep_cols).sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)
test_df  = test_df .dropna(subset=keep_cols).sort_values(["Date", "SecuritiesCode"]).reset_index(drop=True)

train_df["Date"] = pd.to_datetime(train_df["Date"])
test_df ["Date"] = pd.to_datetime(test_df ["Date"])

# 1⃣  Map SecuritiesCode to integer ids based on the UNION of train & test
all_codes = pd.concat([train_df["SecuritiesCode"],
                       test_df ["SecuritiesCode"]]).unique()
code2idx = {c: i for i, c in enumerate(all_codes)}

# 2⃣  Map codes and ensure no NaNs (safe for torch.tensor)
train_df["CodeIdx"] = train_df["SecuritiesCode"].map(code2idx).astype(np.int64)
test_df ["CodeIdx"] = test_df ["SecuritiesCode"].map(code2idx).astype(np.int64)

NUM_CODES = len(code2idx)


scaler = StandardScaler()
train_df[FEATURES] = scaler.fit_transform(train_df[FEATURES])
test_df [FEATURES] = scaler.transform(test_df [FEATURES])

# ----------------- SHARPE METRIC -----------------
def calc_spread_return_sharpe(df, portfolio_size=200, toprank_weight_ratio=2):
    def _per_day(day):
        ps = min(portfolio_size, len(day))
        long  = day.nsmallest(ps, "Rank")["Target"]
        short = day.nlargest(ps,  "Rank")["Target"]
        w = np.linspace(toprank_weight_ratio, 1, ps)
        return (long @ w)/w.mean() - (short @ w)/w.mean()
    daily = df.groupby("Date").apply(_per_day)
    return float(daily.mean()/(daily.std() + 1e-8))

# ----------------- DATASET -----------------
class InferDS(Dataset):
    def __init__(self, train_df, test_df, seq_len, feats):
        full = (pd.concat([train_df.assign(_t=0), test_df.assign(_t=1)])
                  .sort_values(["SecuritiesCode", "Date"])
                  .reset_index(drop=True))
        self.full   = full
        self.codes  = full["SecuritiesCode"].values
        self.pos_in_code = full.groupby("SecuritiesCode").cumcount().values
        self.test_idx = np.where(full["_t"].values == 1)[0]
        self.seq_len, self.feats = seq_len, feats

    def __len__(self): return len(self.test_idx)

    def __getitem__(self, i):
        j   = self.test_idx[i]
        row = self.full.iloc[j]

        # position of this row inside its own security’s timeline
        loc = self.pos_in_code[j]

        # slice the previous seq_len rows of *this* security
        start = max(0, loc - self.seq_len)
        mask  = (self.codes == row["SecuritiesCode"])
        hist  = self.full[mask & (self.pos_in_code < loc)]
        hist  = hist.iloc[-self.seq_len:]            # already sorted

        X = hist[self.feats].values.astype(np.float32)
        if X.shape[0] < self.seq_len:                # left‑pad if needed
            pad = np.tile(X[0] if X.size else row[self.feats].values,
                           (self.seq_len - X.shape[0], 1)).astype(np.float32)
            X = np.vstack([pad, X])

        return torch.from_numpy(X), torch.tensor(row["CodeIdx"], dtype=torch.long), i


infer_ds = InferDS(train_df, test_df, SEQ_LEN, FEATURES)
infer_loader = DataLoader(infer_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=os.cpu_count())

# ----------------- MODEL -----------------
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=500):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        pe = pe.unsqueeze(0)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerRegressor(nn.Module):
    def __init__(self, n_feats, n_codes):
        super().__init__()
        self.code_emb = nn.Embedding(n_codes, DMODEL)
        self.input_proj = nn.Linear(n_feats, DMODEL)
        self.pos_enc = PositionalEncoding(DMODEL, max_len=SEQ_LEN)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=DMODEL, nhead=NHEAD, dim_feedforward=D_FF,
            dropout=DROPOUT, batch_first=True
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=NUM_LAYERS)
        self.head = nn.Sequential(
            nn.LayerNorm(DMODEL),
            nn.Linear(DMODEL, 64), nn.ReLU(), nn.Dropout(0.1),
            nn.Linear(64, 1)
        )

    def forward(self, x, code_idx):
        feat = self.input_proj(x)
        feat = self.pos_enc(feat)
        ce = self.code_emb(code_idx).unsqueeze(1).expand(-1, SEQ_LEN, -1)
        h = feat + ce
        h = self.encoder(h)
        return self.head(h[:, -1]).squeeze(-1)

# ----------------- INFERENCE -----------------
# ─────────── Prepare model & load checkpoint ───────────
model = TransformerRegressor(len(FEATURES), NUM_CODES).to(DEVICE)

# 1. load raw checkpoint
ckpt = torch.load(MODEL_PATH, map_location=DEVICE)

# 2. pull out the pretrained embedding weight
old_emb = ckpt["code_emb.weight"]                      # shape [N₀, D]

# 3. get current model's full state dict
model_sd = model.state_dict()

# 4. copy everything except the embedding into model_sd
for k, v in ckpt.items():
    if k != "code_emb.weight" and k in model_sd:
        model_sd[k].copy_(v)

# 5. now handle embedding: carve out the first N₀ rows
new_emb = model_sd["code_emb.weight"]                   # shape [N₁, D], N₁ ≥ N₀
new_emb[: old_emb.size(0), :] = old_emb

# 6. load the patched state dict
model.load_state_dict(model_sd)
model.eval()


preds = np.empty(len(infer_ds), dtype=np.float32)
with torch.no_grad():
    for xb, ci, idx in tqdm(infer_loader, desc="Inferring"):
        xb, ci = xb.to(DEVICE), ci.to(DEVICE)
        out = model(xb, ci).cpu().numpy()
        preds[idx.numpy()] = out

# ----------------- RESULT -----------------
res = test_df.reset_index(drop=True)
res["pred"] = preds
res["Rank"] = res.groupby("Date")["pred"].rank("first", ascending=False) - 1

rmse   = math.sqrt(mean_squared_error(res["Target"], res["pred"]))
mae    = mean_absolute_error(res["Target"], res["pred"])
sharpe = calc_spread_return_sharpe(res)

print(f"Test RMSE  : {rmse:.4f}")
print(f"Test MAE   : {mae:.4f}")
print(f"Test Sharpe: {sharpe:.4f}")

res[["Date", "SecuritiesCode", "Rank"]].to_csv("submission_improved.csv", index=False)
print("✅ submission_improved.csv written")
